# Phase 4 — End-to-End RAG Pipeline (with Ollama)

**Goal:** Glue together everything from Phases 1–3 into a single function that takes a question and a corpus, retrieves the most relevant chunks, and returns a grounded answer with page citations — running entirely on a local Ollama model.

This is the **first real milestone**: the moment the project becomes useful to a human.

```
Question
   |
   v
Embed question  ->  FAISS search  ->  top-k chunks
                                          |
                                          v
                              build prompt (system + context + question)
                                          |
                                          v
                                       Ollama LLM
                                          |
                                          v
                            Answer + parsed [page X] citations
```


## 4.1 — Reload the vector store


In [ ]:
import os, sys, json
from pathlib import Path
from pydantic import BaseModel
from typing import Optional

repo_root = Path.cwd()
while not (repo_root / "app").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

# Reload chunks + vector store from previous notebooks.
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

class VectorStore:
    def __init__(self, dim, embedder):
        self.index = faiss.IndexFlatIP(dim)
        self.metadata: list[dict] = []
        self.embedder = embedder

    def search(self, query: str, k: int = 5) -> list[dict]:
        import numpy as np
        q = self.embedder.encode([query], normalize_embeddings=True).astype("float32")
        scores, idx = self.index.search(q, k)
        out = []
        for i, s in zip(idx[0], scores[0]):
            if i < 0:
                continue
            m = dict(self.metadata[i]); m["score"] = float(s); out.append(m)
        return out

    @classmethod
    def load(cls, dir_path, embedder):
        p = Path(dir_path)
        cfg = json.loads((p / "config.json").read_text())
        store = cls(cfg["dim"], embedder)
        store.index = faiss.read_index(str(p / "index.faiss"))
        store.metadata = [json.loads(l) for l in (p / "metadata.jsonl").read_text().splitlines()]
        return store

store = VectorStore.load("data/vector_store/aiayn", embedder)
print("Loaded vector store:", store.index.ntotal, "vectors")


## 4.2 — The prompt template

A good RAG prompt has four ingredients:

1. **Role / behavior** — what kind of answers you want.
2. **Grounding rule** — tell the model to refuse if context is insufficient.
3. **Context block** — the retrieved chunks, each labeled with its source.
4. **Citation format** — explicit format the model must use.

We make this a Jinja2 template so it lives next to other prompts in `app/prompts/`.


In [ ]:
from jinja2 import Template

RAG_PROMPT = Template(r"""You are a careful research assistant. Answer the user's question using ONLY the context below.
If the context is insufficient, say "I could not find this in the provided documents."

For every factual claim, cite the source page in square brackets like [page 3] or [page 3, 5].
Be precise. Do not invent details, equations, or numbers.

--- CONTEXT ---
{% for c in chunks %}
[chunk {{ loop.index }} | {{ c.source }} | page {{ c.page }}]
{{ c.text }}

{% endfor %}
--- QUESTION ---
{{ question }}

--- ANSWER (with [page X] citations) ---
""")
print(RAG_PROMPT.render(chunks=[{"source": "x.pdf", "page": 1, "text": "demo"}], question="?")[:400])


## 4.3 — The LLM client wrapper

`app/core/llm.py` exposes a single `chat(messages, **opts)` function so the rest of the code does not care which provider is behind it.


In [ ]:
import ollama, time

class OllamaLLM:
    def __init__(self, model: str = "llama3.2:3b", temperature: float = 0.1):
        self.model = model
        self.temperature = temperature

    def chat(self, system: str, user: str) -> dict:
        t0 = time.time()
        resp = ollama.chat(
            model=self.model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            options={"temperature": self.temperature},
        )
        return {
            "text": resp["message"]["content"],
            "model": self.model,
            "latency_s": time.time() - t0,
            "prompt_tokens": resp.get("prompt_eval_count"),
            "completion_tokens": resp.get("eval_count"),
        }

llm = OllamaLLM("llama3.2:3b")
print("LLM client ready:", llm.model)


## 4.4 — The RAG pipeline as one function


In [ ]:
import re

SYSTEM = "You answer questions grounded in the provided context. Always cite pages."

def rag_answer(question: str, store: VectorStore, llm: OllamaLLM, k: int = 5) -> dict:
    hits = store.search(question, k=k)
    user_prompt = RAG_PROMPT.render(chunks=hits, question=question)
    result = llm.chat(system=SYSTEM, user=user_prompt)
    # Parse [page X] citations from the answer.
    cited = sorted({int(m.group(1)) for m in re.finditer(r"\[page (\d+)", result["text"])})
    return {
        "question": question,
        "answer": result["text"],
        "cited_pages": cited,
        "retrieved": [{"page": h["page"], "score": h["score"], "chunk_id": h["chunk_id"]} for h in hits],
        "latency_s": result["latency_s"],
        "model": result["model"],
    }


## 4.5 — Try it


In [ ]:
out = rag_answer("What is multi-head attention and why is it used?", store, llm, k=5)

print("Q:", out["question"])
print("\nA:", out["answer"])
print("\nCited pages:", out["cited_pages"])
print("Retrieved pages (in score order):", [r["page"] for r in out["retrieved"]])
print(f"Latency: {out['latency_s']:.1f}s")


## 4.6 — A few more questions


In [ ]:
questions = [
    "What dataset and tokenization were used for English-to-German training?",
    "How does scaled dot-product attention scale with sequence length?",
    "Why use sinusoidal positional encodings instead of learned ones?",
    "What were the BLEU scores reported on WMT 2014 English-to-French?",
]

for q in questions:
    out = rag_answer(q, store, llm, k=5)
    print(f"Q: {q}")
    print(f"A: {out['answer'][:280]}{'...' if len(out['answer']) > 280 else ''}")
    print(f"   cited={out['cited_pages']}  latency={out['latency_s']:.1f}s\n")


## 4.7 — Observability: log every run

Senior-engineer move. Every answer becomes a structured JSON line we can inspect, replay, and feed into evaluation.


In [ ]:
def log_run(run: dict, log_file: Path = Path("data/logs/runs.jsonl")) -> None:
    log_file.parent.mkdir(parents=True, exist_ok=True)
    with log_file.open("a", encoding="utf-8") as f:
        f.write(json.dumps(run) + "\n")

# Re-run the first question and log it.
out = rag_answer("What is multi-head attention and why is it used?", store, llm)
log_run(out)
print("Logged 1 run. Most recent entries:")
print(Path("data/logs/runs.jsonl").read_text().splitlines()[-1][:300], "...")


## 4.8 — Failure modes to watch for

| Symptom                              | Likely cause                                                 | Fix                                                  |
|--------------------------------------|--------------------------------------------------------------|------------------------------------------------------|
| "I could not find this..." too often | k too low, or chunk size too small                           | Raise `k`, or rerun chunking with `chunk_size=900`.  |
| Confident but wrong answers          | LLM ignoring context                                         | Strengthen system prompt; lower temperature; add a re-ask "is this in the context?" pass. |
| Cites the wrong page                 | Citation parser sees a page number inside the answer text    | Make the model emit citations only as `[page N]`, and post-validate against retrieved pages. |
| Cuts mid-sentence                    | LLM context too small for chunks + question + answer         | Reduce `k`, or summarize chunks before feeding.       |

We will quantify all of these in Notebook 09 with Ragas.


## What you learned

- The standard RAG flow: embed → retrieve → prompt → answer + parse citations.
- A Jinja2 prompt template you can iterate on.
- An LLM client wrapper that returns latency + token counts.
- Why structured JSONL logging is a senior-engineer move.
- Common RAG failure modes and what to tune.

## Exercises

1. Add a `reranker` step: re-score the top-20 chunks with a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) and keep the top-5. Measure latency vs quality.
2. Add a "no context" guard: if the highest retrieval score is below a threshold (say 0.3), short-circuit and answer "I am not confident I can answer from these documents."
3. Make the citation parser also validate that every cited page was actually retrieved.

## Next: [Phase 5 — FastAPI backend](./2026-05-25-phase05-fastapi-backend.ipynb)
